In [ ]:
from google.colab import files
import pandas as pd

# ── 1. Upload your files ──────────────────────────────────────────────────────
# ── 1. Upload master_clean.csv first ─────────────────────────────────────────
print("Step 1: Upload master_clean.csv")
files.upload()

# ── 2. Upload philly_tract_centroids.csv second ───────────────────────────────
print("Step 2: Upload philly_tract_centroids.csv")
files.upload()

# ── 2. Load ───────────────────────────────────────────────────────────────────
master   = pd.read_csv("master_clean.csv")
centroids = pd.read_csv("philly_tract_centroids.csv")

# Ensure join keys are the same type (string, zero-padded to 11 digits)
master["tract_id"]       = master["tract_id"].astype(str).str.zfill(11)
centroids["GEOID10"]     = centroids["GEOID10"].astype(str).str.zfill(11)

# ── 3. Filter master to 2023 only ─────────────────────────────────────────────
master_2023 = master[master["year"] == 2023].copy()
print(f"master_clean 2023 rows: {len(master_2023)}")
print(f"centroids rows:         {len(centroids)}")

# ── 4. Merge ──────────────────────────────────────────────────────────────────
hex_data = master_2023.merge(
    centroids[["GEOID10", "lat", "lon"]],
    left_on="tract_id",
    right_on="GEOID10",
    how="inner"
)

print(f"Merged rows: {len(hex_data)}")
print(f"Unmatched master tracts: {len(master_2023) - len(hex_data)}")

# ── 5. Keep only the columns the hex map needs ────────────────────────────────
hex_data = hex_data[[
    "tract_id",
    "tract_label",
    "lat",
    "lon",
    "majority_race",
    "median_ratio",
    "median_sale_price",
    "median_assessed_value",
    "assessment_direction",
    "pct_black",
    "pct_white",
    "pct_hispanic",
    "median_income",
    "n_sales",
]].copy()

# ── 6. Filter out low-sale tracts (same threshold as other charts) ────────
hex_data = hex_data[
    (hex_data["n_sales"] >= 10) &
    (hex_data["median_ratio"] > 0.3) &
    (hex_data["majority_race"].notna()) &
    (hex_data["median_sale_price"] < 600000) &
    (hex_data["majority_race"] != "None")
]

print(f"Rows after filtering: {len(hex_data)}")
print(hex_data.head(3))

# ── 7. Save & download ────────────────────────────────────────────────────────
hex_data.to_csv("hex_map_data.csv", index=False)
files.download("hex_map_data.csv")
print("Done! hex_map_data.csv downloaded.")


Step 1: Upload master_clean.csv


Saving master_clean.csv to master_clean.csv
Step 2: Upload philly_tract_centroids.csv


Saving philly_tract_centroids.csv to philly_tract_centroids.csv
master_clean 2023 rows: 49
centroids rows:         384
Merged rows: 43
Unmatched master tracts: 6
Rows after filtering: 29
      tract_id tract_label        lat        lon majority_race  median_ratio  \
2  42101002400    Tract 24  39.936773 -75.159513         White      0.980685   
5  42101003600    Tract 36  39.927933 -75.192024         Black      1.044800   
7  42101005500    Tract 55  39.906902 -75.247501         Black      1.009827   

   median_sale_price  median_assessed_value assessment_direction  pct_black  \
2           400000.0               395900.0       Under-assessed   5.829404   
5           190000.0               203900.0        Over-assessed  63.715656   
7           145000.0               159350.0        Over-assessed  80.543426   

   pct_white  pct_hispanic  median_income  n_sales  
2  75.503643      4.243463        84732.0      689  
5  14.129961      3.416194        43688.0      983  
7   6.210577    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! hex_map_data.csv downloaded.
